# M03 · Getting seismic data: STP, stations, time windows, responses

**Part 1 · week 2 · about 6 h core + 1 h stretch**

## 한국어 요약

이 모듈에서는 지진 파형 자료를 직접 받아서 쓸 수 있는 상태로 만드는 과정을 배웁니다. 관측망 코드(KS는 기상청, KG는 한국지질자원연구원), 관측소 코드, 채널 코드(HHZ, HGZ, ELZ의 대역, 계기, 성분)와 표본화율을 읽고, STP 관측소 목록에 오늘의 관측소 목록보다 관측소가 더 많은 까닭(폐쇄된 관측소)을 확인합니다. 개인 STP 계정과 클라이언트를 안전하게 두는 곳(~/.sgtl, 절대 커밋하거나 공유하지 않음)과 'sgtl data' 명령으로 자료를 받는 순서(먼저 시험 실행)를 익힌 뒤, 어떤 시각 표기로 주어져도 UTC 시간 창을 계산하는 함수, 주어진 날짜에 반경 안에서 운영 중이던 관측소를 고르는 함수, 내려받은 SAC 파일이 빠짐없이 왔는지 세는 함수를 작성합니다. 안병석(부경대학교)의 StationMetadata에서 사건 시각에 유효한 기간의 계기 응답을 찾아 단계별 이득으로 전체 감도를 계산하고, 한 기록에서 계기 응답을 제거해 규모와 거리에 맞는 진폭인지 확인합니다. 마지막으로 동료가 자른 시간 창에 자료가 하나도 없는 까닭을 찾아 바로잡고, 각 기록의 포락선으로 그 증거를 보입니다.

## What you will be able to do

- Read network, station and channel codes (band, instrument, component) and explain why the STP roster lists more stations than today's station list
- Write the time window of a record in UTC from an origin time given in any time zone, and list the stations within a radius that were running on a given day
- Keep a personal STP account private, fetch a dataset with a dry run first, and check what arrived: stations per event and band, and stations that miss a component
- Find the response epoch valid at an event time, compute the overall sensitivity from the stage gains, remove the response from one record and check its amplitude against magnitude and distance
- Find out why a colleague's time window holds no data, fix it, and prove the fix with envelope evidence from the records

## Before you start

- Modules: M01, M02, M04
- Reading: Ahn, Kang & Jung (2024), the English abstract and the figures; Havskov & Alguacil (2016), the chapter on instrument correction
- Needs: stp
- Key terms (see the glossary): STP (파형 전송 프로그램), credentials (계정 정보, 인증 정보), file permissions (파일 권한), dry run (드라이 런, 시험 실행), seismogram (지진 기록, 지진파형), station (관측소), network code (관측망 코드), station code (관측소 코드), retired station (폐쇄된 관측소, 운영 종료 관측소), channel (채널), SEED id (SEED 식별자), location code (위치 코드), band code (대역 코드), instrument code (계기 코드), component (성분), sampling rate (표본화율, 샘플링 레이트), broadband seismometer (광대역 지진계), short-period seismometer (단주기 지진계), accelerometer (가속도계), SAC file (SAC 파일), FDSN (국제 디지털 지진관측망 연맹), UTC (협정 세계시), KST (한국 표준시), naive datetime (나이브 시각, 시간대 정보가 없는 시각), aware datetime (인식 시각, 시간대 정보가 있는 시각), time window (시간 창, 시간 윈도우), station epoch (관측소 운영 기간, 에포크), epicentral distance (진앙 거리), hypocentral distance (진원 거리), great-circle distance (대원 거리), completeness check (자료 완전성 점검), instrument response (계기 응답), response stage (응답 단계), stage gain (단계 이득), counts (카운트, 디지털 값), digitizer (디지타이저, 아날로그-디지털 변환기), overall sensitivity (전체 감도), polarity reversal (극성 반전), StationXML (관측소 메타데이터 XML 형식), response removal (계기 응답 제거), pre-filter (사전 필터), ground motion (지반 운동), Wood-Anderson seismograph (우드-앤더슨 지진계), local magnitude (ML; 국지 규모), envelope (포락선), analytic signal (해석 신호), signal-to-noise ratio (SNR; 신호 대 잡음비), P wave (P파), S wave (S파), travel time (주시)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M03.yaml](../../../.sgtl/criteria/M03.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M03.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"              # the STP dataset of this module (data/manifests/stp_datasets.yaml)
EVENT_ID = "20190721020418"         # one event folder: its UTC origin time as YYYYMMDDhhmmss (the ML 3.9 event)
EPICENTER = (36.50, 128.10)         # latitude, longitude of the Sangju earthquakes (KMA), degrees
DEPTH_KM = 14.0                     # focal depth of that event (KMA), km
RADIUS_KM = 100.0                   # the dataset holds the stations within this epicentral distance
BEFORE_S, AFTER_S = 30.0, 470.0     # every record runs from 30 s before to 470 s after the origin (UTC)
DEMO_STATION = "KS.SAJB"            # Part 6: the station whose record loses its instrument response
PRE_FILT = (0.3, 0.6, 30.0, 40.0)   # Part 6: corner frequencies (Hz) of the taper used in the response removal
TRAP_ROW = 1                        # Part 7: the row of event_catalog.csv that the colleague used
FIGURE_DIR = "figures"              # your figure files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M03")

import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from obspy import UTCDateTime, read, read_inventory
from obspy.geodetics import gps2dist_azimuth
from obspy.signal.filter import envelope

DATA = st.data.root() / DATASET                 # this module's dataset, outside the repository
SAC_ROOT = DATA / "stp_download" / "SAC"        # one folder per event, one folder per band inside it
META = st.data.root() / "metadata"              # StationMetadata, installed by: bin/sgtl-install metadata --apply
Path(FIGURE_DIR).mkdir(exist_ok=True)


def need(path, what):
    """Stop with a plain message when a file or folder that a cell needs is missing; return it as a Path."""
    if not Path(path).exists():
        raise FileNotFoundError(f"{what} is missing ({path}). Part 1 shows how to get it.")
    return Path(path)

## How this notebook works

The same five steps as in M01, M02 and M04, for every task: read the **Contract**, write a **prediction**, replace
`raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer the
**Explain** prompt in 3-5 sentences of your own. `st.hint("M03-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on fresh made-up inputs whose answers are known from
how they were built; the checks of tasks 3, 4 and 5 also run your code on the real data and never print the
real answers. `st.reroll("M03-T1")` draws new made-up inputs.

This module needs two things outside the repository: the **waveforms** of six Sangju earthquakes, which you
fetch with your own STP account (Part 1), and the **station metadata** of Byeong-Seok Ahn, which the installer
fetches (Part 6). The synthetic checks of tasks 1 and 2 need no downloads, but running the notebook in order
needs the fetched dataset: complete the preparation in Part 1 before continuing to Part 2.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M03.md`. Never paste your STP settings, data files or file paths of other people into an AI tool.

## Part 1 · Your STP account and this module's data (about 25 min)

A **seismogram** (glossary `seismogram`) is a record of ground motion against time at one station. The
seismograms of the Korean national networks come from **STP** (glossary `stp`), a program that sends your
requests to a data center and saves the answers as files. You use it with a **personal account**: a user name
and a password that the lab arranges for you. Together they are your **credentials** (glossary `credentials`):
whoever has them can download data in your name.

**Where your STP files live.** You receive two private things from your mentor: the account and the STP client
program `stp-client.pl` (a Perl script). Keep both in the hidden folder `~/.sgtl/` in your home folder, which is
**outside** the repository:

```bash
mkdir -p ~/.sgtl && chmod 700 ~/.sgtl          # only you may open this folder
cp <the client you received> ~/.sgtl/stp-client.pl
nano ~/.sgtl/credentials.env                   # write the two lines below, save, quit
chmod 600 ~/.sgtl/credentials.env ~/.sgtl/stp-client.pl
```

`~/.sgtl/credentials.env` holds two lines (your values instead of the parts in angle brackets):

```text
STP_USER=<your STP user name>
STP_PASS=<your STP password>
```

`chmod 600` sets the **file permissions** (glossary `file_permissions`) so that only you can read the file: on a
shared server, a file with the default permissions can be read by every other user. Rules that have no
exceptions: never commit these files, never copy them into the repository, never paste their contents into a
chat, an e-mail or an AI tool, and never lend your account. If you think someone saw your password, tell your
mentor the same day. The course's `sgtl data` command refuses a credentials file that other users can read, and
it runs the client only when its SHA-256 fingerprint matches the one in `install/checksums.txt`.

The cell below checks that the two files exist and that only you can read them. It never reads their contents.

In [ ]:
private_dir = Path.home() / ".sgtl"
for name in ("credentials.env", "stp-client.pl"):
    item = private_dir / name
    if not item.is_file():
        print(f"~/.sgtl/{name}: not found. Follow the steps above when you have your account.")
        continue
    mode = item.stat().st_mode & 0o777
    verdict = "only you can read it: good" if mode & 0o077 == 0 else "OTHER USERS CAN READ IT: run chmod 600 on it"
    print(f"~/.sgtl/{name}: found, permissions {mode:03o}, {verdict}")

**Fetching a dataset.** The course keeps a list of datasets in `data/manifests/stp_datasets.yaml`: for each one,
the events (copied from the KMA catalog, in UTC), the networks, the largest station distance and the record
window. In a terminal (not in this notebook), the `sgtl data` command works through that list, and like every
course script it does nothing until you add `--apply` (a **dry run**, glossary `dry_run`):

```bash
sgtl data list                          # the datasets, their sizes, and what you already have
sgtl data fetch sangju2019              # dry run: the plan; nothing is downloaded
sgtl data fetch sangju2019 --apply      # downloads with your account, through PocketQuake (a few minutes)
sgtl data verify sangju2019             # file counts, damaged files and SHA-256 fingerprints
```

Before fetching, run `bin/sgtl-install tools metadata` in a server terminal, read its dry-run plan, then repeat
with `--apply`. This installs PocketQuake (which performs the download) and the station metadata used in Part 6.
If your copy of the
tutorial does not know `sgtl data` yet, update it (`bin/update-from-upstream`, a dry run first) or ask your
mentor. Everything lands in your data folder (`st.data.root()`), never in the repository. The dataset entry of
this module:

In [ ]:
manifest = yaml.safe_load(st.data.repo_path("data", "manifests", "stp_datasets.yaml").read_text(encoding="utf-8"))
entry = manifest["datasets"][DATASET]
print(entry["title"], "| networks", entry["networks"], "| stations within", entry["station_radius_km"], "km",
      "| window", entry["window_s"], "s around each origin")
pd.DataFrame(entry["events"])

**The file layout.** After the download, the dataset folder looks like this:

```text
<data folder>/sangju2019/
  event_catalog/event_catalog.csv      the six events, one row each (look at the times: Part 3)
  station_table/KS_station.csv         every station STP knows in network KS, with coordinates
  station_table/KG_station.csv         the same for network KG
  stp_download/stp_batch.txt           the requests that were sent to STP
  stp_download/SAC/<event>/<band>/<start>.<NET>.<STA>.<CHA>.sac    one file per channel and event
  MANIFEST.sha256                      a fingerprint of every file, for sgtl data verify
```

Each **SAC file** (glossary `sac_file`; Helffrich et al. 2013; formats in Havskov & Ottemöller 2010) holds one
channel of one record: a header with the station code, the channel, the start time and the sample interval, then
the samples. The event folders are named after the event's origin time in UTC (`YYYYMMDDhhmmss`), and the file
names start with the record's start time in UTC. The cell below shows what you have.

In [ ]:
if not SAC_ROOT.is_dir():
    print(f"No waveforms yet in {DATA}. Fetch them first (above); tasks 1 and 2 work without them.")
else:
    for event_dir in sorted(p for p in SAC_ROOT.iterdir() if p.is_dir()):
        per_band = {b.name: len(list(b.glob("*.sac"))) for b in sorted(event_dir.iterdir()) if b.is_dir()}
        print(event_dir.name, per_band)
    batch_lines = (DATA / "stp_download" / "stp_batch.txt").read_text(encoding="utf-8").splitlines()
    print(len(batch_lines), "lines in stp_batch.txt; the first three:")
    print("\n".join(batch_lines[:3]))

In `stp_batch.txt`, a `dir` line chooses the folder for the next answers, and a `win` line asks for one window:
`win KS DUSB HH_ 2018/02/24,21:16:07.000000 500s` means network KS, station DUSB, all components of the HH
channels, starting at that UTC time, for 500 seconds. Part 5 compares these requests with what arrived. The file
keeps the requests of the latest STP session only: if you fetched the dataset in several sessions, the events of
the earlier ones are missing from it.

## Part 2 · Networks, stations and channels (about 25 min)

Every record is named by four codes, written together as a **SEED id** (glossary `seed_id`) like `KS.SAJB..HHZ`:

- the **network code** (glossary `network_code`): here `KS`, the national network of the Korea Meteorological
  Administration (KMA), or `KG`, the network of the Korea Institute of Geoscience and Mineral Resources (KIGAM);
- the **station code** (glossary `station_code`): three to five letters or digits for one site;
- the **location code** (glossary `location_code`): tells apart two sensors of the same kind at one station; it is
  empty for every channel in this dataset (hence the two dots in a row);
- the **channel code** (glossary `channel`): three letters.

The three letters of a channel code follow the SEED convention of the FDSN (glossary `fdsn`):

1. the **band code** (glossary `band_code`) tells the sampling rate (glossary `sampling_rate`) and the sensor's
   period range: `H` and `E` are 80-250 samples per second, `H` for a **broadband seismometer** (glossary
   `broadband_seismometer`, which records periods of tens of seconds) and `E` for a **short-period seismometer**
   (glossary `short_period_seismometer`, about 1 s and shorter); `B` and `S` are the same two kinds at 10-80
   samples per second;
2. the **instrument code** (glossary `instrument_code`) tells the kind of sensor: `H` a high-gain seismometer,
   `L` a low-gain seismometer, and `G`, in the KS and KG networks, an **accelerometer** (glossary
   `accelerometer`), a sensor that stays on scale in strong shaking (other networks write `N` for it);
3. the **component** (glossary `component`): `Z` vertical, `N` north, `E` east.

So `HHZ` is the vertical channel of a broadband seismometer at 100 samples per second, `HGZ` the vertical
channel of an accelerometer, and `ELZ` that of a short-period seismometer. Never trust the letters alone: Part 6
reads the sensor and its units from the station metadata.

In [ ]:
event_dir = need(SAC_ROOT / EVENT_ID, f"The event folder {EVENT_ID}")
names = pd.DataFrame([f.name.split(".") for f in sorted(event_dir.glob("*/*.sac"))],
                     columns=["start", "network", "station", "channel", "suffix"])
print(f"{len(names)} files in event {EVENT_ID}; files per channel code:")
print(names["channel"].value_counts().sort_index().to_dict())
print("stations per network:", names.groupby("network")["station"].nunique().to_dict())
headers = read(str(event_dir / "*" / "*KG.CHS.*.sac"), headonly=True)
for tr in headers:
    print(tr.id, f"{tr.stats.sampling_rate:g} samples/s, {tr.stats.npts} samples, starts {tr.stats.starttime}")

**Two station lists.** The repository ships today's station list (`st.data.stations()`, with start dates and no
end dates), and the dataset holds the **roster** that STP returned with its `sta` command: every station it
knows, also those that no longer run. A station that stopped recording is a **retired station** (glossary
`retired_station`); its code stays in old data. When a station moves a short distance, it often gets a new code
with a digit, such as `CHJ2` after `CHJ`.

In [ ]:
today = st.data.stations()
roster = pd.concat([pd.read_csv(need(DATA / "station_table" / f"{net}_station.csv", "The STP station roster"))
                    for net in ("KS", "KG")], ignore_index=True)
roster = roster.rename(columns={"Network": "network", "Code": "station", "Latitude": "latitude",
                                "Longitude": "longitude", "Elevation": "elevation_m"})
in_today = set(today["network"] + "." + today["station"])
in_roster = set(roster["network"] + "." + roster["station"])
print(f"today's list: {len(in_today)} stations; STP roster: {len(in_roster)} stations")
print(f"in the roster only: {len(in_roster - in_today)}; in today's list only: {len(in_today - in_roster)}")
print("a few roster-only codes:", sorted(in_roster - in_today)[:12])
twins = roster.groupby("station")["network"].nunique()
print("station codes used by both networks:", sorted(twins[twins > 1].index))

Two lessons for later: a station code alone does not name a station (`CHS` and `BRD` exist in both networks, at
different sites), so always keep the network code with it; and a list of today's stations misses stations that
recorded the old earthquakes you study.

## Part 3 · Time windows in UTC (about 10 min, then task 1)

A record is cut in a **time window** (glossary `time_window`) around the origin time: this dataset keeps 30 s
before the origin (the background noise) and 470 s after it (all the waves of a nearby earthquake). Data centers,
file names and headers count time in **UTC** (glossary `utc`). Korean reports, and the catalog file of this
dataset, give the time in **KST** (glossary `kst`), UTC + 9 h. Compare the two:

In [ ]:
kst_rows = pd.read_csv(need(DATA / "event_catalog" / "event_catalog.csv", "The dataset's event catalog"))
display(kst_rows)
print("event folders (UTC):", sorted(p.name for p in SAC_ROOT.iterdir() if p.is_dir()))
kma = st.data.sequence(DATASET)
print(kma.loc[kma["event_id"].isin([e["event_id"] for e in entry["events"]]),
              ["event_id", "origin_utc", "origin_kst", "ml"]].to_string(index=False))

The rows of `event_catalog.csv` have no zone at all: nothing in the file says that 11:04:18 is Korean time.
The KMA list in the repository (KMA catalog, KOGL Type 1) keeps both clocks (M01, M04). A time without a zone
is a **naive** time (glossary `naive_datetime`); a time with its zone names one moment (glossary
`aware_datetime`).

### M03-T1 · Time windows in UTC · core · about 35 min

**Goal:** write `utc_window(origin_utc, before_s, after_s)`: the start and the end of a record window around an
origin time, written in UTC.

**Given:** origin times reach you in many spellings: `"2019-07-21T02:04:18Z"`, `"2019-07-21T11:04:18+09:00"`
(the same moment in KST), text without a zone, a pandas `Timestamp` with or without a zone, a Python `datetime`
with a zone, or an ObsPy `UTCDateTime` (always UTC).

**Contract:**
- Inputs: `origin_utc` in any spelling above: with a zone or an offset it is that moment, without one it is UTC;
  `before_s`, `after_s`: seconds before and after the origin (0 or more, fractions allowed).
- Output: a tuple `(start, end)` of two strings in ISO 8601 form, in UTC with the zone written out (`Z` or
  `+00:00`): `start` is `before_s` seconds before the origin, `end` is `after_s` seconds after it, correct to
  the millisecond.
- Example: `utc_window("2019-07-21T11:04:18+09:00", 30, 470)` gives
  `("2019-07-21T02:03:48+00:00", "2019-07-21T02:12:08+00:00")` (the same moments written with `Z`, or with
  fractions of a second, are fine too).

**Done when** `st.check("M03-T1", func=utc_window)` passes on 12 fresh origin times in all the spellings above,
with fractions of a second.

**Stuck?** `st.hint("M03-T1", 1)`

In [ ]:
# Before you code: the first row of event_catalog.csv says 2018-02-25 06:16:37 (KST). In UTC, when do its record
# window (30 s before, 470 s after) start and end, and on which date?
st.predict("M03-T1", """

""")

In [ ]:
def utc_window(origin_utc, before_s, after_s):
    """Start and end of a record window around an origin time, as UTC text.

    origin_utc : origin time as text, pandas Timestamp, datetime or ObsPy UTCDateTime; with a zone or offset it
                 is that moment, without one it is UTC
    before_s   : float, seconds before the origin (>= 0)
    after_s    : float, seconds after the origin (>= 0)
    returns    : (start, end), two ISO 8601 strings in UTC with the zone written out (Z or +00:00)
    """
    # Step 1: turn origin_utc into one moment in UTC, whatever its spelling (an ObsPy UTCDateTime included).
    # Step 2: move back before_s seconds and forward after_s seconds.
    # Step 3: write both moments as ISO text that keeps the zone.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T1: write your code here")

In [ ]:
st.check("M03-T1", func=utc_window)

Your function on the dataset's events (UTC times from the manifest), written the way STP wants them, and
compared with the requests that were really sent:

In [ ]:
sent = {line.split()[4] for line in (DATA / "stp_download" / "stp_batch.txt").read_text().splitlines()
        if line.startswith("win ")}
for ev in entry["events"]:
    start, end = utc_window(ev["origin_utc"], BEFORE_S, AFTER_S)
    as_stp = pd.Timestamp(start).strftime("%Y/%m/%d,%H:%M:%S.%f")
    length = (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds()
    print(f"{ev['event_id']}  ML {ev['ml']}  {start} to {end} ({length:.0f} s)  STP start {as_stp}  "
          f"sent: {as_stp in sent}")
first = kst_rows.iloc[0].astype(int)
written = f"{first.Year}-{first.Month:02d}-{first.Day:02d} {first.Hour:02d}:{first.Minute:02d}:{first.Second:02d}"
start, end = utc_window(written, BEFORE_S, AFTER_S)
print(f"the first catalog row, {written}, read as if it were UTC: {start} to {end}")

**Explain (3-5 sentences):** Why must the two window strings carry their zone? The last line above reads the first
catalog row as if it were UTC: how far off is that window, and what would STP have sent back for it? Why does the
first event's folder name show another date than its catalog row?

## Part 4 · Which stations were running? (about 15 min, then task 2)

A station's life is split into **station epochs** (glossary `station_epoch`): periods with the same site,
instruments and settings. Every change (a new sensor, a new digitizer, a move) ends one epoch and starts the
next. Station metadata lists the epochs of every channel with a start and an end date; an end date far in the
future (here 2099-12-31 or 9999-12-31) means the epoch has not ended.

The cell below reads the station metadata of every roster station near Sangju and keeps, for each station, the
first start date and the last end date of its channels (a simplification: a gap between two epochs is ignored).
Part 6 explains the metadata itself. The **epicentral distance** (glossary `epicentral_distance`) is the
great-circle distance (glossary `great_circle_distance`) from the epicenter to the station along the surface.

In [ ]:
def station_epochs(roster_df, center, half_width_deg, xml_dir):
    """One row per roster station near `center` that has a StationXML file: first start and last end date.

    roster_df      : DataFrame with network and station columns (the STP roster)
    center         : (latitude, longitude) in degrees
    half_width_deg : keep stations within this many degrees of the center in latitude and in longitude
    xml_dir        : folder of NET.STA.xml files
    returns        : DataFrame network, station, latitude, longitude, start_date, end_date (dates without a time
                     zone; end_date NaT = still running), and the list of stations without a file
    """
    lat0, lon0 = center
    near = roster_df[(roster_df["latitude"] - lat0).abs().le(half_width_deg)
                     & (roster_df["longitude"] - lon0).abs().le(half_width_deg)]
    rows, no_file = [], []
    for net, sta in zip(near["network"], near["station"]):
        path = Path(xml_dir) / f"{net}.{sta}.xml"
        if not path.is_file():
            no_file.append(f"{net}.{sta}")
            continue
        site = read_inventory(str(path))[0][0]
        starts = [c.start_date for c in site.channels]
        ends = [c.end_date for c in site.channels]
        last = None if any(e is None or e.year >= 2099 for e in ends) else max(ends)
        rows.append({"network": net, "station": sta, "latitude": site.latitude, "longitude": site.longitude,
                     "start_date": pd.Timestamp(min(starts).date),
                     "end_date": pd.NaT if last is None else pd.Timestamp(last.date)})
    return pd.DataFrame(rows), no_file


epochs, without_file = station_epochs(roster, EPICENTER, 1.2, need(META / "stationxml", "StationMetadata"))
print(f"{len(epochs)} stations with metadata near Sangju; {len(without_file)} roster stations have no file")
print("with an open end (none, 2099 or 9999):", int(epochs["end_date"].isna().sum()))
epochs.head()

### M03-T2 · Stations running on a given day · core · about 45 min

**Goal:** write `stations_within(stations_df, lat, lon, radius_km, on_date)`: the stations within a radius that
were running on a given day.

**Given:** a station table with one row per station epoch and at least the columns `network`, `station`,
`latitude`, `longitude` (degrees) and `start_date`, usually also `end_date` (pandas dates without a time zone).
A missing end date (`NaT`), or no `end_date` column at all (as in today's list), means still running. The table
may hold other columns, and one station can have several rows (one per epoch, for example before and after a
move).

**Contract:**
- Inputs: `stations_df`; `lat`, `lon`: the center in degrees; `radius_km`: in kilometers; `on_date`: a calendar
  day as text `"YYYY-MM-DD"` or as a pandas `Timestamp` without a zone.
- A row is running on `on_date` when its `start_date` is on or before that day and its `end_date` is missing or
  on or after that day: the first and the last day both count.
- Output: a DataFrame with the rows that are running on `on_date` and whose epicentral distance from
  (`lat`, `lon`) is at most `radius_km`; all columns of `stations_df` plus a column `distance_km` (the
  great-circle distance on a sphere of radius 6371 km; a geodesic library is fine too); sorted by `distance_km`,
  nearest first. When no row qualifies: an empty DataFrame with those columns.
- Do not change `stations_df`.

**Done when** `st.check("M03-T2", func=stations_within)` passes on 8 fresh made-up tables (epochs that start or
end on the day itself, stations without an end date, tables without an `end_date` column, moved stations, one
station code in two networks).

**Stuck?** `st.hint("M03-T2", 1)`

In [ ]:
# Before you code: for the ML 3.9 event of 2019-07-21, will today's station list or the metadata epochs give more
# stations within 100 km? Which stations will each of them get wrong?
st.predict("M03-T2", """

""")

In [ ]:
def stations_within(stations_df, lat, lon, radius_km, on_date):
    """Station epochs running on a given day within a radius of a point, nearest first.

    stations_df : DataFrame with network, station, latitude, longitude (degrees), start_date and optionally
                  end_date (dates without a zone; NaT or no column = still running), maybe other columns
    lat, lon    : float, the center in degrees
    radius_km   : float, the largest distance in km
    on_date     : the day, as text "YYYY-MM-DD" or a pandas Timestamp without a zone
    returns     : DataFrame of the running rows within radius_km (all columns plus distance_km), sorted by
                  distance_km; stations_df itself is not changed
    """
    # Step 1: which rows are running on that day (both ends count; no end date means still running)?
    # Step 2: the distance of every row from the center, in km.
    # Step 3: keep the rows that pass both tests, add distance_km, sort.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T2: write your code here")

In [ ]:
st.check("M03-T2", func=stations_within)

Your function on the two lists, for the day of the ML 3.9 event, compared with the stations that sent data:

In [ ]:
event_day = pd.Timestamp(EVENT_ID[:8])
lat0, lon0 = EPICENTER
by_today = stations_within(today, lat0, lon0, RADIUS_KM, event_day)
by_epochs = stations_within(epochs, lat0, lon0, RADIUS_KM, event_day)
with_data = set(names["network"] + "." + names["station"])
for label, table in (("today's list", by_today), ("metadata epochs", by_epochs)):
    ids = set(table["network"] + "." + table["station"])
    print(f"{label}: {len(ids)} stations running within {RADIUS_KM:g} km; {len(ids & with_data)} of them sent data;"
          f" {len(with_data - ids)} stations with data are not on it: {sorted(with_data - ids)}")
print(f"{len(with_data)} stations sent data for {EVENT_ID}")

Compare the requested radius (dashed circle) with the stations that actually supplied records. A metadata
epoch covering the event day does not guarantee a usable record. Different station identifiers can share
nearly the same location, so a red open triangle may overlap a blue triangle. Check the station lists above
before interpreting an overlapping symbol as one contradictory record.

In [ ]:
import pygmt

region = [lon0 - 1.35, lon0 + 1.35, lat0 - 1.05, lat0 + 1.05]
quiet = by_epochs[~(by_epochs["network"] + "." + by_epochs["station"]).isin(with_data)]
sent_rows = roster[(roster["network"] + "." + roster["station"]).isin(with_data)]
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):
    fig_map = pygmt.Figure()
    fig_map.basemap(region=region, projection="M10c", frame=["af", f"+tStations of event {EVENT_ID}"])
    fig_map.coast(land="gray92", water="white", shorelines="0.25p,gray40")
    fig_map.plot(data=np.array([[lon0, lat0, 2.0 * RADIUS_KM]]), style="E-k", pen="0.6p,gray30,-")  # 100-km circle
    fig_map.plot(data=quiet[["longitude", "latitude"]].to_numpy(), style="i0.22c", pen="0.6p,red3",
                 label="Listed in metadata; no record+S0.2c")
    fig_map.plot(data=sent_rows[["longitude", "latitude"]].to_numpy(), style="t0.22c", fill="dodgerblue",
                 pen="0.3p,black", label="Record received+S0.2c")
    fig_map.plot(data=np.array([[lon0, lat0]]), style="a0.45c", fill="gold", pen="0.5p,black", label="Epicenter+S0.3c")
    fig_map.legend(position="jBR+o0.2c", box="+gwhite+p0.5p")
    fig_map.basemap(map_scale=st.plot.scale_bar_spec(region, km=50))
fig_map.savefig(Path(FIGURE_DIR) / "m03_stations_map.png", dpi=200)
fig_map.show()

**Explain (3-5 sentences):** Which list found more of the stations that sent data, and why does today's list miss
some of them? Name two reasons why a station that the metadata calls running can still have sent nothing. Why
do both ends of an epoch count as running, and what goes wrong with `end_date >= day` when the end date is
missing?

## Part 5 · Did everything arrive? (about 10 min, then task 3)

A **completeness check** (glossary `completeness_check`) compares what you asked for with what you got. STP
answers a request for a station that had no data (no such band, an outage, a station not yet installed) with
nothing, and it does not say so. Count before you compute: a station with only one or two of its three
components can break a later step that expects all three.

### M03-T3 · What arrived? · core · about 45 min

**Goal:** write `completeness_table(sac_root)`: count the stations per event and band in a downloaded SAC tree,
and list the stations that miss a component.

**Given:** a folder `sac_root` with one folder per event (named by the event's UTC origin time), and in each event
folder one folder per band (`HH`, `HG`, `EL`, ...) with SAC files named `<start>.<NET>.<STA>.<CHA>.sac`. Some
event folders lack a band folder or are empty, some band folders are empty, and other files (notes, hidden
files) can lie anywhere. The same station code can appear in two networks.

**Contract:**
- Input: `sac_root`, text or a `pathlib.Path`.
- Output: a tuple `(counts, gaps)`:
  - `counts`: a DataFrame with one row per event folder (index: the folder names as text, sorted) and one column
    per band folder found anywhere in the tree (column names: the band folder names, sorted); each value is the
    number of stations (network and station together) with at least one SAC file in that band folder of that
    event, 0 when there are none;
  - `gaps`: a DataFrame with the columns `event`, `station` (as `"NET.STA"`), `band` and `missing`: one row for
    each event, station and band that has some but not all of the components `E`, `N` and `Z` (the last letter
    of the channel code), where `missing` lists the absent letters in alphabetical order (`"N"` or `"EZ"`); rows
    sorted by event, then band, then station; an empty DataFrame with these four columns when nothing is missing.
- Only files whose names end in `.sac` and that lie directly in a band folder count. Read the codes from the file
  names. Do not change anything in the tree.

**Done when** `st.check("M03-T3", func=completeness_table)` passes on made-up SAC trees that the check writes
into a temporary folder (answers known from how they were built) and on your real Sangju tree: there the check
counts nothing itself; your function must run without changing the tree, and on a copy of one band folder from
which the check removed one file, your own result must change by exactly that file.

**Stuck?** `st.hint("M03-T3", 1)`

In [ ]:
# Before you code: from Part 1, how many HH stations do you expect for the ML 3.9 event, and do you expect any
# station with a missing component in this dataset?
st.predict("M03-T3", """

""")

In [ ]:
def completeness_table(sac_root):
    """Stations per event and band in a SAC tree, and the stations that miss a component.

    sac_root : str or Path of the folder with one folder per event, one folder per band inside each
    returns  : (counts, gaps): counts = DataFrame events x bands of station counts (ints, 0 when none);
               gaps = DataFrame event, station ("NET.STA"), band, missing (absent letters of E, N, Z)
    """
    # Step 1: walk the event folders and their band folders; collect, per event and band, the components found
    #         for every network-and-station.
    # Step 2: fill the counts table (every event folder a row, every band folder a column, 0 where nothing).
    # Step 3: list the stations whose components are not all three, with the absent letters.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T3: write your code here")

In [ ]:
st.check("M03-T3", func=completeness_table)

Your function on the real tree, and the requests of `stp_batch.txt` counted per event and band:

In [ ]:
counts, gaps = completeness_table(SAC_ROOT)
requested = {}
for line in (DATA / "stp_download" / "stp_batch.txt").read_text().splitlines():
    if line.startswith("dir "):
        folder = Path(line.split(maxsplit=1)[1])
        key = (folder.parent.name, folder.name)
    elif line.startswith("win "):
        requested[key] = requested.get(key, 0) + 1
asked = pd.Series(requested).unstack().reindex(index=counts.index, columns=counts.columns)
print("stations that sent data, per event and band:")
print(counts.to_string())
print("\nshare of the requested stations that sent data:")
print((counts / asked).round(2).to_string())
print(f"\n{len(gaps)} event-station-band combinations miss a component")
display(gaps.head())

**Explain (3-5 sentences):** What share of the requests came back, per band, and why is it so far from 1 even for
the largest event? Why must a station be counted with its network code? What would happen to a later step that
rotates the two horizontal components if a station had only `E` and `Z`?

## Part 6 · Instrument responses (about 25 min, with task 4 in the middle)

A seismometer does not write ground motion into the file. It writes **counts** (glossary `counts`): whole numbers
from a **digitizer** (glossary `digitizer`), the device that measures the sensor's voltage many times per second.
The **instrument response** (glossary `instrument_response`) says how the recording chain turns **ground
motion** (glossary `ground_motion`; velocity in m/s for a seismometer, acceleration in m/s² for an
accelerometer) into counts. It is written as a list of **response stages** (glossary `response_stage`):

1. the sensor turns ground motion into volts, for example 1500 V per m/s;
2. the digitizer turns volts into counts, for example 419 430 counts per volt;
3. digital filters shape the signal and reduce the sampling rate, usually with a gain of 1.

Each stage has a **stage gain** (glossary `stage_gain`): how much it multiplies its input at a reference
frequency. Multiplying the gains of all stages gives the **overall sensitivity** (glossary
`overall_sensitivity`) in counts per m/s (or per m/s²): for the broadband seismometers of this dataset, from
about 6 × 10⁸ to 2.5 × 10⁹ counts for a ground velocity of 1 m/s, in the band of frequencies where the response is
flat. Outside that band the response
is smaller, so dividing counts by one number works only inside it; removing the response properly divides by the
whole response, frequency by frequency (Havskov & Alguacil 2016; Scherbaum 2001).

By the StationXML convention, a negative stage gain says that the channel records the ground motion upside down,
a **polarity reversal** (glossary `polarity_reversal`). The sign matters for first-motion polarities (M20, M51);
the sensitivity, a size, has none. Metadata are claims about an instrument, though, and a minus sign is easy to get
wrong: below you meet a file that disagrees with itself, and the records decide which of its claims is right.

**Station metadata.** The responses of the KS and KG stations come from **StationMetadata**, integrated by
**Byeong-Seok Ahn (Pukyong National University)** from the records of KMA, KIGAM and the Korea Institute of
Nuclear Safety (Ahn, Kang & Jung 2024). The files are StationXML (glossary `stationxml`), one per station, and the
installer puts them in your data folder. Cite Ahn et al. (2024) whenever you use them; the acknowledgment file
says how:

In [ ]:
acknowledgment = need(META / "ACKNOWLEDGMENT.txt", "StationMetadata (run: bin/sgtl-install metadata --apply)")
print("\n".join(acknowledgment.read_text(encoding="utf-8").splitlines()[:14]))

One station's channel epochs, read with ObsPy (Beyreuther et al. 2010; Krischer et al. 2015; the ObsPy tutorial
has more examples). The table shows,
for every epoch of the vertical channels, the gain of each stage and the stored overall sensitivity:

In [ ]:
chs = read_inventory(str(META / "stationxml" / "KG.CHS.xml"))
epoch_rows = []
for channel in chs[0][0].channels:
    if channel.code.endswith("Z"):
        epoch_rows.append({"channel": channel.code, "start": str(channel.start_date)[:19],
                           "end": str(channel.end_date)[:19], "sensor": channel.sensor.description,
                           "samples/s": channel.sample_rate,
                           "stage gains": [s.stage_gain for s in channel.response.response_stages],
                           "units": channel.response.response_stages[0].input_units,
                           "stored sensitivity": channel.response.instrument_sensitivity.value})
pd.DataFrame(epoch_rows)

KG.CHS kept its accelerometer (`HG`, `BG`) from 2009 on, but its broadband channels (`HH`, `BH`) start a new
epoch on 2014-11-20, and the gain of the first stage changes sign. A record from 2013 and one from 2019 need
different epochs. `B` channels are the same sensors recorded at 20 samples per second.

### M03-T4 · Counts per meter per second · core · about 45 min

**Goal:** write `sensitivity_at(inventory, seed_id, time)`: the overall sensitivity of one channel at one moment,
and the units of the ground motion it refers to.

**Given:** an ObsPy `Inventory` (from `read_inventory`) with one or more networks, stations, channels and channel
epochs. Each channel has `.code`, `.location_code`, `.start_date`, `.end_date` (`None` when it has not ended) and
`.response.response_stages`, a list of stages with `.stage_gain`, `.input_units` and `.output_units`.

**Contract:**
- Inputs: `inventory`; `seed_id`, text `"NET.STA.LOC.CHA"` (`LOC` can be empty: `"KG.CHS..HHZ"`); `time`, an
  ObsPy `UTCDateTime`.
- An epoch covers `time` when `start_date <= time` and (`end_date` is `None` or `time <= end_date`).
- Output: a tuple `(value, ground_units)`: `value` is the size (always positive) of the product of the gains of all
  stages of the epoch that covers `time`, in counts per unit of ground motion; `ground_units` is the input units
  of its first stage, as written in the file (for example `"M/S"` or `"M/S**2"`). When no epoch of that channel
  covers `time`, return `None`.
- Work it out from the stages: do not read the stored overall value (`instrument_sensitivity`), not even through
  `print`, and do not call `get_response`. The check records both.

**Done when** `st.check("M03-T4", func=sensitivity_at)` passes on fresh made-up inventories (the answers come from
ObsPy's `get_response`) and on five random channels of your Sangju data in StationMetadata (the check never
prints their values).

**Stuck?** `st.hint("M03-T4", 1)`

In [ ]:
# Before you code: about how many counts per m/s do you expect for KG.CHS..HHZ in 2019, and how much smaller for
# its accelerometer per m/s² (look at the table above)? What do you return for KG.CHS..HHZ in 2005?
st.predict("M03-T4", """

""")

In [ ]:
def sensitivity_at(inventory, seed_id, time):
    """Overall sensitivity of one channel at one moment, from the gains of its response stages.

    inventory : ObsPy Inventory
    seed_id   : text "NET.STA.LOC.CHA" (LOC may be empty)
    time      : ObsPy UTCDateTime
    returns   : (value, ground_units): value > 0 in counts per unit of ground motion, ground_units the input units
                of the first stage (e.g. "M/S"); None when no epoch of that channel covers time
    """
    # Step 1: split the id into its four codes.
    # Step 2: find the channel epoch with those four codes that covers time.
    # Step 3: multiply the gains of all its stages; keep the size; read the units of the first stage.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T4: write your code here")

In [ ]:
st.check("M03-T4", func=sensitivity_at)

Your function on every vertical channel of the ML 3.9 event. The column `negative first gain` counts the channels
whose first stage carries a minus sign (by the convention above, reversed channels; the records test that claim
below):

In [ ]:
origin = UTCDateTime(f"{EVENT_ID[:8]}T{EVENT_ID[8:]}")
z_rows = []
for path in sorted(event_dir.glob("*/*Z.sac")):
    _, net, sta, cha, _ = path.name.split(".")
    inv = read_inventory(str(META / "stationxml" / f"{net}.{sta}.xml"))
    found = sensitivity_at(inv, f"{net}.{sta}..{cha}", origin)
    first_gain = None
    if found is not None:
        first_gain = inv.select(channel=cha, time=origin)[0][0][0].response.response_stages[0].stage_gain
    z_rows.append({"id": f"{net}.{sta}..{cha}", "counts per unit": None if found is None else found[0],
                   "units": None if found is None else found[1],
                   "negative first gain": first_gain is not None and first_gain < 0})
sens = pd.DataFrame(z_rows)
print(sens.groupby(sens["id"].str[-3:-1]).agg(**{"channels": ("id", "size"),
                                                 "negative first gain": ("negative first gain", "sum"),
                                                 "median": ("counts per unit", "median")}).to_string())
print("units:", sens["units"].value_counts().to_dict())

**Removing the response** (glossary `response_removal`). ObsPy's `remove_response` divides the spectrum of the
record by the whole response, frequency by frequency. Two safety measures keep it from blowing up where the
response is tiny: the mean is removed and the ends are tapered, and a **pre-filter** (glossary `pre_filter`,
`PRE_FILT`) fades out the frequencies below the second and above the third corner. Compare it with dividing the
counts by your one number:

In [ ]:
net, sta = DEMO_STATION.split(".")
raw = read(str(event_dir / "HH" / f"*.{net}.{sta}.HHZ.sac"))[0]
station_xml = read_inventory(str(META / "stationxml" / f"{net}.{sta}.xml"))
value, units = sensitivity_at(station_xml, raw.id, raw.stats.starttime)
rough = (raw.data - raw.data.mean()) / value
velocity = raw.copy()
velocity.detrend("demean")
velocity.taper(0.05)
velocity.remove_response(inventory=station_xml, output="VEL", pre_filt=PRE_FILT)
seconds = raw.times(reftime=origin)
fig, ax = plt.subplots(figsize=(8.0, 3.2))
ax.plot(seconds, rough * 1e6, lw=0.6, color="0.55", label="Counts divided by the sensitivity")
ax.plot(seconds, velocity.data * 1e6, lw=0.6, color="tab:blue", label="Response removed")
ax.set_xlim(-5, 40)
ax.set_xlabel("Seconds after the origin (UTC)")
ax.set_ylabel("Ground velocity (µm/s)")
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "m03_response_removal.png", dpi=150)
print(f"{raw.id}: {value:.4g} counts per {units}; peak |velocity| {np.abs(velocity.data).max() * 1e6:.1f} µm/s "
      f"(response removed) and {np.abs(rough).max() * 1e6:.1f} µm/s (counts / sensitivity)")
print("correlation of the two traces:", round(float(np.corrcoef(rough, velocity.data)[0, 1]), 3))

For channels with a negative first-stage gain (KS.SAJB is one), the C library behind `remove_response` (evalresp)
prints a warning that the computed and the reported sensitivities differ by more than 5 percent: the stages give
the sensitivity with a minus sign, the stored value has none. It goes on with the stored, positive value, so the
corrected trace keeps the sign of the counts, as the figure shows.

So the file disagrees with itself: its stages say "reversed", its stored sensitivity says "not reversed", and the
SAC pole-zero file of the same collection (`KS_KG_metadata_1.0.2.SACPZ`) has a positive constant for this epoch
although its header lists the gain as −1500. Which part is right? Only the records can tell. A reversed channel
shows first motions opposite to those of a second sensor at the same site and of its neighbours at nearly the
same azimuth from the source, for every earthquake. For this ML 3.9 event we tested every broadband channel of
the dataset with a negative first gain: each one moves first the same way as the accelerometer at the same site
(whose gains are all positive) and as its neighbours at a similar azimuth, and the same way as KIGAM's published
mechanism predicts. Their counts are not upside down; in M20 you test this with your own picks. In this
collection the sign of a stage gain is therefore not a reliable polarity flag.
First motions (M20, M51) take their sign from the counts; when a channel looks reversed, test it against the
records of several earthquakes before you turn anything around, and tell the metadata's author what you found.

**A sanity check of the amplitude.** Does the amplitude fit an ML 3.9 earthquake at this distance? Richter's
**local magnitude** (glossary `local_magnitude`; Richter 1935) uses the largest swing of a **Wood-Anderson
seismograph** (glossary `wood_anderson_seismograph`), an old instrument that we can simulate from the ground
displacement. The IASPEI standard (Bormann & Dewey 2014), built on Hutton & Boore (1987) for Southern California,
is $M_L = \log_{10} A + 1.11 \log_{10} R + 0.00189\,R - 2.09$, with $A$ the largest Wood-Anderson amplitude in nm
on a horizontal component and $R$ the **hypocentral distance** (glossary `hypocentral_distance`) in km. Korea has
its own distance correction (Sheen et al. 2018), so expect agreement within a few tenths, not better.

In [ ]:
WOOD_ANDERSON = {"poles": [-6.283 + 4.712j, -6.283 - 4.712j], "zeros": [0j, 0j], "gain": 1.0, "sensitivity": 1.0}
ml_rows = []
for north_file in sorted(event_dir.glob("HH/*HHN.sac")):
    pair = read(str(north_file)) + read(str(north_file).replace("HHN.sac", "HHE.sac"))
    net, sta = pair[0].stats.network, pair[0].stats.station
    station_xml = read_inventory(str(META / "stationxml" / f"{net}.{sta}.xml"))
    peaks = []
    for tr in pair:
        tr.detrend("demean")
        tr.taper(0.05)
        tr.remove_response(inventory=station_xml, output="DISP", pre_filt=PRE_FILT)
        tr.simulate(paz_simulate=WOOD_ANDERSON)
        peaks.append(np.abs(tr.data).max() * 1e9)          # nm
    epi_km = gps2dist_azimuth(lat0, lon0, pair[0].stats.sac.stla, pair[0].stats.sac.stlo)[0] / 1000.0
    hypo_km = float(np.hypot(epi_km, DEPTH_KM))
    ml_rows.append({"station": f"{net}.{sta}", "epi_km": epi_km, "amplitude_nm": max(peaks),
                    "ml": np.log10(max(peaks)) + 1.11 * np.log10(hypo_km) + 0.00189 * hypo_km - 2.09})
ml_table = pd.DataFrame(ml_rows).sort_values("epi_km", ignore_index=True)
catalog_ml = kma.loc[kma["origin_utc"] == pd.Timestamp(origin.datetime, tz="UTC"), "ml"].iloc[0]
print(ml_table.round(2).to_string())
print(f"median of {len(ml_table)} stations: ML {ml_table['ml'].median():.2f} "
      f"(spread {ml_table['ml'].quantile(0.25):.2f}-{ml_table['ml'].quantile(0.75):.2f}); KMA catalog: ML {catalog_ml}")

**Explain (3-5 sentences):** Why did your function need the event time, and what would a record from 2013 at
KG.CHS get wrong if it used today's epoch? In the response-removal figure, how do the two traces differ for this
station, and what did the removal do with its negative first-stage gain? Does the amplitude pass the sanity check,
and what would a missing factor of 1000 (for example nm against µm) have done to the magnitude?

## Part 7 · A window without an earthquake (about 10 min, then task 5)

A colleague wants a figure of the largest Sangju earthquake at every broadband station, from 10 s before to 60 s
after the origin. The colleague takes the origin time from the dataset's own catalog file, picks the event folder
with the same row number (the folders and the rows are in the same order) and cuts the window. The cell runs
without an error and reports that not one of the broadband records holds data in the window. The colleague
concludes that the broadband stations missed the earthquake. Can that be true?

Your evidence will be the **envelope** (glossary `envelope`) of each record: a smooth curve that follows the size
of the wiggles, computed as the size of the **analytic signal** (glossary `analytic_signal`). The analytic signal
pairs each sample with the same sample of a copy of the record in which every frequency is shifted by a quarter
of a cycle; a sine and its shifted copy never cross zero together, so the size of the pair, √(x² + shifted²),
follows the height of the wiggles smoothly (ObsPy: `obspy.signal.filter.envelope`; SciPy:
`abs(scipy.signal.hilbert(x))`). The ratio of the envelope's peak to the background noise is a
**signal-to-noise ratio** (glossary `signal_to_noise_ratio`). For a nearby earthquake the peak usually belongs to
the **S wave** (glossary `s_wave`), which arrives after the **P wave** (glossary `p_wave`) at a **travel time**
(glossary `travel_time`) that grows with distance, about 1 s for every 3.5 km.

### M03-T5 · A window without an earthquake · core · about 60 min

**Goal:** find out what went wrong in the colleague's cell below, fix it with `corrected_window(row, before_s,
after_s)`, and prove the fix with envelope evidence from your own function `envelope_evidence(...)`.

**Given:** the colleague's cell (run it, do not edit it). It defines `trap_row` (a row of `event_catalog.csv`:
`Year`, `Month`, `Day`, `Hour`, `Minute`, `Second`, `Latitude`, `Longitude`, `Depth`, `Magnitude`), `origin`
(the colleague's origin time), `trap_dir` (the event folder) and `records` (its HHZ records).

**Contract:**
- `corrected_window(row, before_s, after_s)`: `row` is a pandas Series or a dict with the fields of one row of
  `event_catalog.csv` (numbers, maybe written as floats such as `2019.0`; `Second` may have a fraction); `before_s`,
  `after_s` as in task 1. Output: like task 1, a tuple `(start, end)` of two ISO strings in UTC with the zone
  written out, for the window around the moment of the earthquake that the row describes (its origin time in the
  KMA catalog). The check's made-up rows are written the same way as the rows of `event_catalog.csv`, at dates
  anywhere in the KMA list (1978 to today).
- `envelope_evidence(records, row, origin, wrong_origin, before_s, after_s)`: `records` is an ObsPy Stream of
  vertical records (station coordinates in the SAC header: `tr.stats.sac.stla`, `tr.stats.sac.stlo`; any
  sampling rate), `row` a catalog row as above, `origin` the true origin and `wrong_origin` the colleague's origin
  (both ObsPy UTCDateTime), `before_s` and `after_s` the window around each origin. Output: a DataFrame with one
  row per record, nearest station first, and the columns
  - `station` (`"NET.STA"`), `distance_km` (epicentral distance from the row's latitude and longitude to the
    station),
  - `n_wrong`: the number of samples of the record inside the colleague's window (`wrong_origin - before_s` to
    `wrong_origin + after_s`),
  - `t_peak_s`: the time of the envelope's peak inside your window (`origin - before_s` to `origin + after_s`),
    in seconds after `origin`,
  - `peak_ratio`: that peak divided by the median of the envelope over the first 20 s of the record.

  Compute the envelope of the whole record after removing its mean, with no filter, and only then look inside the
  windows. Leave `records` unchanged (work on copies).
- `evidence`: the table of your `envelope_evidence` for `records` and `trap_row`, with the true origin from your
  `corrected_window` and the colleague's `origin`, 10 s before and 60 s after.

**Done when** `st.check("M03-T5", answer={"corrected_window": corrected_window, "envelope_evidence":
envelope_evidence, "evidence": evidence, "row": trap_row})` passes: your `corrected_window` on fresh made-up rows
and your `envelope_evidence` on made-up records (answers known from how they were built), and then your
`evidence` table must be exactly what your own function returns for the real records of the row. The check
computes nothing of its own on the real records.

**Stuck?** `st.hint("M03-T5", 1)`

In [ ]:
# The colleague's cell. Run it; do not edit it.
trap_row = kst_rows.iloc[TRAP_ROW]
origin = UTCDateTime(int(trap_row["Year"]), int(trap_row["Month"]), int(trap_row["Day"]), int(trap_row["Hour"]),
                     int(trap_row["Minute"]), int(trap_row["Second"]))
trap_dir = sorted(p for p in SAC_ROOT.iterdir() if p.is_dir())[TRAP_ROW]
records = read(str(trap_dir / "HH" / "*.HHZ.sac"))
window = records.slice(origin - 10.0, origin + 60.0)
print(f"event {trap_dir.name}: origin {origin}, ML {trap_row['Magnitude']}")
print(f"{len(records)} broadband vertical records; {len(window)} of them hold data from 10 s before to 60 s after")
print("Conclusion: the broadband stations did not record this earthquake.")

In [ ]:
# Before you code: when (in UTC) do the records in trap_dir start and end, and where does the colleague's window
# fall? If the records do hold the earthquake, at what time after the true origin do you expect the envelope peak
# at a station 50 km away?
st.predict("M03-T5", """

""")

In [ ]:
def corrected_window(row, before_s, after_s):
    """Time window around the origin of one row of event_catalog.csv, in UTC.

    row      : pandas Series or dict with Year, Month, Day, Hour, Minute, Second (numbers; Second may have a
               fraction) and maybe other fields
    before_s : float, seconds before the origin (>= 0)
    after_s  : float, seconds after the origin (>= 0)
    returns  : (start, end), two ISO 8601 strings in UTC with the zone written out
    """
    # Step 1: build the clock reading of the row, and say which clock it was read on.
    # Step 2: turn it into the window in UTC (your task 1 can help).
    # YOUR CODE HERE
    raise NotImplementedError("M03-T5: write your code here")

In [ ]:
def envelope_evidence(records, row, origin, wrong_origin, before_s, after_s):
    """Envelope evidence for every record: distance, samples in the colleague's window, peak time and size.

    records      : obspy Stream of vertical records with tr.stats.sac.stla and tr.stats.sac.stlo
    row          : pandas Series or dict with Latitude and Longitude (degrees)
    origin       : obspy UTCDateTime, the true origin
    wrong_origin : obspy UTCDateTime, the colleague's origin
    before_s, after_s : float, seconds before and after each origin
    returns      : DataFrame with columns station, distance_km, n_wrong, t_peak_s, peak_ratio, nearest first
    """
    # Step 1: for every record, count its samples in the colleague's window.
    # Step 2: envelope of a copy of the record minus its mean; noise level; peak inside your window.
    # Step 3: distance to the station; one row per record; sort.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T5: write your code here")

In [ ]:
# Build `evidence` (see the Contract) with your two functions, for the records of trap_dir.
# YOUR CODE HERE
raise NotImplementedError("M03-T5: write your code here")
print(evidence.round(2).to_string())

In [ ]:
st.check("M03-T5", answer={"corrected_window": corrected_window, "envelope_evidence": envelope_evidence,
                           "evidence": evidence, "row": trap_row})

The evidence as a figure: the peak times against distance, with the travel times of P and S waves in a simple
crust (6.0 and 3.5 km/s, straight rays from the catalog depth):

In [ ]:
d_axis = np.linspace(0.0, RADIUS_KM, 101)
hypo_axis = np.hypot(d_axis, float(trap_row["Depth"]))
fig, ax = plt.subplots(figsize=(5.8, 4.0))
ax.plot(d_axis, hypo_axis / 6.0, color="0.5", lw=1.0, ls="--", label="P, 6.0 km/s")
ax.plot(d_axis, hypo_axis / 3.5, color="0.2", lw=1.0, label="S, 3.5 km/s")
dots = ax.scatter(evidence["distance_km"], evidence["t_peak_s"], c=np.log10(evidence["peak_ratio"]), s=28,
                  cmap="viridis", edgecolors="black", linewidths=0.4, zorder=3)
fig.colorbar(dots, ax=ax, label="log10 of the peak-to-noise ratio")
ax.set_xlabel("Epicentral distance (km)")
ax.set_ylabel("Envelope peak, s after the origin")
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(Path(FIGURE_DIR) / "m03_envelope_evidence.png", dpi=150)
print(f"stations: {len(evidence)}; samples in the colleague's window: {int(evidence['n_wrong'].sum())}; "
      f"smallest peak-to-noise ratio {evidence['peak_ratio'].min():.0f}")

**Explain (3-5 sentences):** What exactly was wrong in the colleague's cell, by how much, and why did it run without
an error? Which numbers of your table prove that the broadband stations did record the earthquake, and how do the
peak times change with distance? Name one other place in a project where the same slip would go wrong quietly.
For the first row of the catalog, what changes besides the hour?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M03")

### Going further (optional, about 60 min, not checked)

1. Set `DATASET = "suncheon2019"`, `EVENT_ID`, `EPICENTER` and `DEPTH_KM` to one of its events (the manifest
   lists them) after you have fetched it, and run Parts 3-6 again. Which bands does the Suncheon area have?
2. Compute the Wood-Anderson magnitude of the other five Sangju events. How does the spread between stations
   change for the smallest events, and why?
3. Find the stations of the dataset whose first-stage gain changed sign between two epochs, and the dates. For
   one of them, compare the first motions of an earthquake before the change and one after it with those of a
   neighbour at a similar azimuth: did the records change, or only the metadata? Why does this matter for
   first-motion polarities (M51)?

Next: M10 works with these seismograms: counts and ground motion, filters, and rotation to radial and transverse.